# P1 · Motor de diferenciación automática desde cero

**Nivel Intermedio** · Trabajo práctico P1 · Se entrega al inicio de la clase de la semana 8 · Tiempo estimado: 4–5 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/P1_autograd_desde_cero.ipynb)

En la semana 7 escribimos a mano la vuelta (*backward pass*) de una red fija: cada fórmula pensada para esa red. Si le agregas una capa o cambias ReLU por otra cosa, hay que volver a derivar todo. En este trabajo lo automatizas. Vas a construir `Value`: un número que **recuerda de qué operación salió**. Cada operación sabe su derivada local, y una sola llamada, `loss.backward()`, recorre el grafo hacia atrás y llena el gradiente de todos los números de los que depende la pérdida, sea cual sea la cuenta.

Es la idea de *micrograd*, el motor de unas cien líneas de Andrej Karpathy, y es lo mismo que hace `autograd` de PyTorch (semana 8) con arreglos en vez de números sueltos. Con tu motor vas a entrenar una red neuronal sobre el conjunto *two moons*, y compararla con una línea base, con la regresión logística y con sklearn.

**Objetivos.** Al terminar podrás:

1. Programar la derivada local de cada operación como una función que se guarda junto al resultado, y recorrer un grafo en orden topológico para aplicar la regla de la cadena.
2. Verificar un motor de gradientes contra la derivada numérica y contra PyTorch.
3. Armar un MLP con tu motor y entrenarlo con descenso de gradiente, comparado con una línea base.
4. Explicar con tus números por qué un motor que trabaja número por número es lento, y qué pasa si olvidas reiniciar los gradientes.

**Prerrequisitos.** Clases en Python (semana 1), derivadas y regla de la cadena (semana 4), MLP y ReLU (semana 6), backpropagation (semana 7). Si faltaste a alguna, el "Repaso rápido" de abajo trae lo necesario. No hace falta haber entregado ningún trabajo anterior.

**Cómo entregar.** Antes de escribir nada: *Archivo → Guardar una copia en GitHub*, repositorio `Alto-Rendimiento-IA/entregas-<tu-usuario>`, rama `main`, ruta **`P1/P1_autograd_desde_cero.ipynb`**, mensaje `Empezar P1`. Trabaja en la copia que se abre y guarda ahí después de cada ejercicio. Lo que esté en tu repositorio al inicio de la clase de la semana 8 es tu entrega.

**Uso de IA.** Puedes preguntarle a una IA qué significa un error o cómo funciona un `set` de Python. No le pidas el código de `Value`, de `backward`, de las capas ni del entrenamiento: ese es el ejercicio. Si quieres ver el video de Karpathy que se recomendó en la semana 7, míralo **antes** de empezar; después ciérralo y escribe tu motor sin él.

## Preparación

Ejecuta esta celda primero. Carga las librerías y los datos, y los separa en entrenamiento y prueba. **Esa partición queda fija** para todos los modelos.

PyTorch solo aparece en las celdas de verificación, como referencia para comparar tus gradientes. No tienes que escribir nada con PyTorch: es el tema de la semana 8.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_moons
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

SEED = 0
torch.set_default_dtype(torch.float64)

X_moons, y_moons = make_moons(n_samples=200, noise=0.15, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.25, stratify=y_moons, random_state=SEED)

# Tu motor trabaja con números de Python, uno por uno: pasamos los arreglos a listas.
Xl_train, Xl_test = X_train.tolist(), X_test.tolist()
yl_train, yl_test = y_train.tolist(), y_test.tolist()


def numerical_derivative(f, x, h=1e-6):
    """Diferencia centrada (semana 4)."""
    return (f(x + h) - f(x - h)) / (2 * h)


fig, ax = plt.subplots(figsize=(5, 3.5))
for c, color in [(0, "tab:blue"), (1, "tab:orange")]:
    ax.scatter(*X_train[y_train == c].T, s=18, color=color, edgecolor="white", label=f"clase {c}")
ax.set(title="Two moons: el conjunto de entrenamiento", xlabel="$x_1$", ylabel="$x_2$")
ax.legend(fontsize=8)
plt.show()
print("entrenamiento:", X_train.shape, " prueba:", X_test.shape)

## Repaso rápido

- **Grafo computacional.** Una cuenta dibujada como nodos (operaciones) y flechas (números). Ejemplo: $\text{total} = (\text{precio} + \text{envío}) \times \text{cantidad}$, con $3$, $2$ y $4$: el subtotal es 5 y el total 20.
- **La regla de la vuelta.** Se empieza en la salida con gradiente 1 y se recorre el grafo al revés. En cada nodo: **gradiente de una entrada = gradiente que llega × derivada local**. En el ejemplo, el total le pasa $1 \times 4 = 4$ al subtotal y $1 \times 5 = 5$ a la cantidad; el subtotal le pasa $4 \times 1 = 4$ al precio y al envío.
- **Patrones.** La suma copia el gradiente a sus entradas. La multiplicación le da a cada entrada el gradiente por **la otra** entrada. ReLU deja pasar el gradiente si su entrada era positiva, y si no, lo corta.
- **Ramas.** Si un número se usa en dos lugares, recibe gradiente por los dos caminos y **los gradientes se suman**. Ejemplo: en $x \cdot x$, cada "copia" de $x$ recibe la otra, y en total $x + x = 2x$.
- **Derivadas que vas a necesitar** (semana 4): $\dfrac{d}{dx} x^k = k\,x^{k-1}$, $\;\dfrac{d}{dx} e^x = e^x$, $\;\dfrac{d}{dx} \ln x = \dfrac{1}{x}$, $\;\dfrac{d}{dx} \text{ReLU}(x) = 1$ si $x > 0$ y $0$ si no.
- **Descenso de gradiente.** $\theta \leftarrow \theta - \eta\, \partial L / \partial \theta$, con $\eta$ la tasa de aprendizaje.
- **MLP.** Cada neurona calcula $z = \sum_j w_j x_j + b$; las ocultas aplican ReLU y la de salida no. Para dos clases, la salida $z$ se convierte en probabilidad con la sigmoide $\sigma(z) = 1/(1 + e^{-z})$ y se predice 1 si $z > 0$.

### Una herramienta de Python: funciones que recuerdan

Una función definida **dentro** de otra recuerda las variables de afuera, aunque la de afuera ya haya terminado. Se llama *closure* (clausura). Tu motor la usa para guardar, junto a cada resultado, la receta para repartir su gradiente:

In [ ]:
def make_greeting(name):
    def greet():
        return f"hola, {name}"   # greet recuerda name
    return greet


say_hi = make_greeting("Ana")
print(say_hi())  # make_greeting ya terminó, pero say_hi todavía sabe quién es Ana

## Ejercicio 1: un número que recuerda de dónde salió

Un `Value` guarda cinco cosas:

| Atributo | Qué es |
|---|---|
| `data` | el número |
| `grad` | $\partial L / \partial$ este número, donde $L$ es la salida final. Empieza en 0 y se llena en la vuelta |
| `_prev` | los `Value` de los que salió (sus "hijos" en el grafo) |
| `_op` | la operación que lo produjo, como texto (`"+"`, `"*"`), solo para mostrarla |
| `_backward` | una función sin argumentos que reparte `self.grad` entre sus hijos. Para un número que no salió de ninguna operación, no hace nada |

**1a.** Completa `__add__` y `__mul__`. Cada una debe:

1. Calcular el resultado, `out = Value(<el número>, (self, other), "<op>")`.
2. Definir **adentro** una función `_backward()` que sume a `self.grad` y a `other.grad` lo que les toca: gradiente que llega (`out.grad`) por la derivada local. Usa `+=`, no `=`: si un `Value` se usa en dos lugares, sus gradientes se suman.
3. Guardarla en `out._backward` y devolver `out`.

Gracias a `__radd__` y `__rmul__`, que ya están escritas, también funcionan `2 + v`, `2 * v` y `sum(lista_de_values)`.

In [ ]:
class Value:
    """Un número que recuerda de qué operación salió, para poder calcular gradientes hacia atrás."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = tuple(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4g}, grad={self.grad:.4g})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO 1a
        raise NotImplementedError

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO 1a
        raise NotImplementedError

    def __radd__(self, other):  # para 2 + v y para sum(...)
        return self + other

    def __rmul__(self, other):  # para 2 * v
        return self * other

In [ ]:
a, b = Value(3.0), Value(-2.0)
s = a + b
p = a * b
assert s.data == 1.0 and p.data == -6.0, "revisa el número que calcula cada operación"
assert set(s._prev) == {a, b} and set(p._prev) == {a, b}, "out debe recordar sus dos hijos en _prev"
assert s._op == "+" and p._op == "*", 'la suma se marca "+" y la multiplicación "*"'
assert (a + 1).data == 4.0 and (2 * a).data == 6.0 and (1 + a).data == 4.0, "falla con números que no son Value"

s.grad = 5.0
s._backward()
assert (a.grad, b.grad) == (5.0, 5.0), f"la suma copia el gradiente: esperaba (5, 5), hay {(a.grad, b.grad)}"
p.grad = 10.0
p._backward()
assert b.grad == 5.0 + 10.0 * 3.0, "la multiplicación da a b el gradiente que llega por a (y se suma a lo que ya tenía)"
assert a.grad == 5.0 + 10.0 * -2.0, "¿usaste += ? a ya tenía 5 de la suma; ahora debe sumar 10 · (−2)"
print("✓ suma y multiplicación, ida y derivada local")

**Guarda en GitHub** con el mensaje `Ejercicio 1 resuelto`.

## Ejercicio 2: la vuelta completa

Ya cada nodo sabe repartir su gradiente. Falta llamar a los `_backward` **en el orden correcto**: un nodo solo puede repartir cuando ya recibió **todo** su gradiente, es decir, después de todos los nodos que lo usan.

Piensa en vestirte: las medias van antes que los zapatos. Hacia adelante, un nodo va después de sus hijos; en la vuelta, al revés. Un **orden topológico** (*topological order*) es una lista donde cada nodo aparece después de todos sus hijos. Se arma con una búsqueda en profundidad:

```
construir(v):
    si v no está en visitados:
        agregar v a visitados
        construir(cada hijo de v)
        agregar v al final de la lista
```

**2a.** Completa `backward(self)`:

1. Arma la lista en orden topológico empezando en `self`, con un `set` de visitados.
2. Pon `self.grad = 1.0` (la salida cambia exactamente lo que cambia la salida).
3. Recorre la lista **al revés** y llama a `_backward()` de cada nodo.

La función se "pega" a la clase con `Value.backward = backward`, así no tienes que reescribir la clase. Después, cualquier `Value` tiene el método: `loss.backward()`.

In [ ]:
def backward(self):
    # TODO 2a
    raise NotImplementedError


Value.backward = backward

`trace` imprime un grafo como un árbol, de la salida hacia las hojas: cada nodo con su operación, su valor y su gradiente. Úsala para depurar: si un gradiente sale mal, mira dónde empieza a fallar.

In [ ]:
def trace(v, indent=0):
    """Imprime el grafo de v como un árbol: cada nodo con su operación, su valor y su gradiente, y debajo sus hijos.
    Un nodo usado dos veces aparece dos veces. Sirve para grafos chicos."""
    print(" " * indent + f"{v._op or 'hoja'}   data = {v.data:.4f}   grad = {v.grad:.4f}")
    for child in v._prev:
        trace(child, indent + 4)


price, shipping, quantity = Value(3.0), Value(2.0), Value(4.0)
total = (price + shipping) * quantity
total.backward()
trace(total)

In [ ]:
assert (price.grad, shipping.grad, quantity.grad) == (4.0, 4.0, 5.0), "la compra: precio 4, envío 4, cantidad 5"

x = Value(3.0)
y = x * x
y.backward()
assert x.grad == 6.0, f"x · x en x = 3 debe dar 6 (las dos ramas se suman); da {x.grad}"


def diamond(a, b, c):
    """b se usa dos veces y d se usa en dos lugares: el orden de la vuelta importa."""
    d = (a + b) * (b + c)
    return d * a + d * c + b


va, vb, vc = Value(2.0), Value(-1.0), Value(4.0)
out = diamond(va, vb, vc)
out.backward()
for v, name, f in [(va, "a", lambda t: diamond(t, -1.0, 4.0)), (vb, "b", lambda t: diamond(2.0, t, 4.0)),
                   (vc, "c", lambda t: diamond(2.0, -1.0, t))]:
    assert math.isclose(v.grad, numerical_derivative(f, v.data), rel_tol=1e-6), f"el gradiente de {name} no coincide con el numérico"
print("✓ backward: la compra, las ramas y un grafo con caminos repetidos")

**Guarda en GitHub** con el mensaje `Ejercicio 2 resuelto`.

## Ejercicio 3: más operaciones

Con suma y multiplicación no alcanza para una red. **3a.** Completa cuatro operaciones más, con el mismo patrón del ejercicio 1 (resultado, `_backward` con `+=`, devolver `out`). Cada una tiene un solo hijo, `self`:

| Función | Calcula | Derivada local |
|---|---|---|
| `value_pow(self, k)` | $x^k$, con `k` un número de Python (no un `Value`) | $k\, x^{k-1}$ |
| `value_exp(self)` | $e^x$ (`math.exp`) | $e^x$: ¡es el propio resultado! |
| `value_log(self)` | $\ln x$ (`math.log`) | $1 / x$ |
| `value_relu(self)` | $\max(x, 0)$ | 1 si $x > 0$, 0 si no |

Usa como `_op` el texto que quieras (`f"**{k}"`, `"exp"`, `"log"`, `"relu"`).

In [ ]:
def value_pow(self, k):
    assert isinstance(k, (int, float)), "el exponente debe ser un número de Python"
    # TODO 3a
    raise NotImplementedError


def value_exp(self):
    # TODO 3a
    raise NotImplementedError


def value_log(self):
    # TODO 3a
    raise NotImplementedError


def value_relu(self):
    # TODO 3a
    raise NotImplementedError


Value.__pow__ = value_pow
Value.exp = value_exp
Value.log = value_log
Value.relu = value_relu

Las demás operaciones **no necesitan derivada propia**: se arman con las que ya tienes, y la vuelta les sale gratis. La resta es sumar el opuesto; dividir es multiplicar por la potencia $-1$; y la sigmoide es $1 / (1 + e^{-x})$. Ejecuta la celda:

In [ ]:
Value.__neg__ = lambda self: self * -1
Value.__sub__ = lambda self, other: self + (-other)
Value.__rsub__ = lambda self, other: (-self) + other
Value.__truediv__ = lambda self, other: self * (other if isinstance(other, Value) else Value(other)) ** -1
Value.__rtruediv__ = lambda self, other: Value(other) * self ** -1
Value.sigmoid = lambda self: 1 / (1 + (-self).exp())

v = Value(0.0)
s = v.sigmoid()
s.backward()
print(f"sigmoide(0) = {s.data}, su derivada = {v.grad}   (la de la semana 4: σ(0)(1 − σ(0)) = 0.25)")

In [ ]:
points = [-1.7, -0.3, 0.4, 2.2]
for name, op, f in [("x ** 3", lambda v: v ** 3, lambda t: t ** 3),
                    ("x ** -1.5", lambda v: v ** -1.5, lambda t: abs(t) ** -1.5),
                    ("exp", lambda v: v.exp(), math.exp),
                    ("log", lambda v: v.log(), lambda t: math.log(abs(t))),
                    ("relu", lambda v: v.relu(), lambda t: max(t, 0.0))]:
    for t in points:
        t = abs(t) if name in ("log", "x ** -1.5") else t
        v = Value(t)
        out = op(v)
        out.backward()
        assert math.isclose(out.data, f(t), rel_tol=1e-9), f"{name}: el valor en {t} está mal"
        assert math.isclose(v.grad, numerical_derivative(f, t), rel_tol=1e-5, abs_tol=1e-8), \
            f"{name}: la derivada en {t} no coincide con la numérica"
print("✓ las cuatro operaciones coinciden con la derivada numérica")

# Una expresión larga, contra PyTorch
vals = [1.5, -0.7, 0.3]
mine = [Value(v) for v in vals]
ref = [torch.tensor(v, requires_grad=True) for v in vals]


def expression(a, b, c, exp, relu):
    d = (a * b + exp(c)) / (a - c) ** 2
    e = (d * d + 1).log() if isinstance(d, Value) else torch.log(d * d + 1)
    return e * relu(a + b) - b / (1 + exp(-a * c)) + a ** 3


out_mine = expression(*mine, Value.exp, Value.relu)
out_ref = expression(*ref, torch.exp, torch.relu)
out_mine.backward()
out_ref.backward()
assert math.isclose(out_mine.data, out_ref.item(), rel_tol=1e-9), "el valor de la expresión no coincide con PyTorch"
for name, m, r in zip("abc", mine, ref):
    assert math.isclose(m.grad, r.grad.item(), rel_tol=1e-9), f"el gradiente de {name} no coincide con PyTorch"
print("✓ una expresión larga: mismos gradientes que PyTorch")

**Guarda en GitHub** con el mensaje `Ejercicio 3 resuelto`.

## Ejercicio 4: la red de la semana 7, sin derivar nada

En la clase de la semana 7 hicimos en papel la vuelta de esta red, con la entrada $x = (2, -1)$ y la respuesta correcta $y = 1$:

$$W_1 = \begin{pmatrix} 1 & 2 \\ -1 & 1 \end{pmatrix} \quad b_1 = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \qquad W_2 = \begin{pmatrix} 2 & -1 \end{pmatrix} \quad b_2 = -0.5$$

$$h = \text{ReLU}(W_1 x + b_1) \qquad p = \sigma(W_2 h + b_2) \qquad L = -\left[\,y \ln p + (1 - y) \ln(1 - p)\,\right]$$

**4a.** Escribe la red con tu motor: cada peso y cada sesgo es un `Value` (ya están creados abajo), la cuenta se escribe con las operaciones de `Value`, y al final llamas a `L.backward()`. Usa `.sigmoid()` y `.log()`. Después guarda los gradientes como números en `dW1` (lista de 2 filas de 2), `db1` (lista de 2), `dW2` (lista de 2) y `db2` (un número).

No derives nada: escribe la ida, y la vuelta la hace tu motor.

In [ ]:
x_net, y_net = [2.0, -1.0], 1.0
W1_v = [[Value(1.0), Value(2.0)], [Value(-1.0), Value(1.0)]]
b1_v = [Value(1.0), Value(0.0)]
W2_v = [Value(2.0), Value(-1.0)]
b2_v = Value(-0.5)

# TODO 4a: la ida, L, y L.backward()
L = None
dW1, db1, dW2, db2 = None, None, None, None  # TODO 4a

In [ ]:
assert isinstance(L, Value), "L debe ser un Value: la pérdida armada con tu motor"
assert math.isclose(L.data, -math.log(1 / (1 + math.exp(-1.5))), rel_tol=1e-9), "revisa la ida: L debería ser −ln σ(1.5)"
assert np.allclose(dW1, [[-0.7297, 0.3649], [0.0, 0.0]], atol=1e-3), "dW1 no coincide con la tabla de la semana 7"
assert np.allclose(db1, [-0.3649, 0.0], atol=1e-3), "db1 no coincide con la tabla de la semana 7"
assert np.allclose(dW2, [-0.1824, 0.0], atol=1e-3), "dW2 no coincide con la tabla de la semana 7"
assert math.isclose(db2, -0.1824, abs_tol=1e-3), "db2 no coincide con la tabla de la semana 7"
print("✓ los mismos números que la tabla de la semana 7, sin derivar nada")

**4b.** En la semana 7 la segunda fila de $\partial L / \partial W_1$ salió cero. ¿Qué línea de tu motor es la responsable de ese cero? Y en `L` usaste `.sigmoid()`, que no tiene derivada propia: ¿cómo obtuvo tu motor su gradiente?

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 4 resuelto`.

## Ejercicio 5: neuronas, capas y una red

Ahora las piezas de una red, con la misma forma que `nn.Module` de PyTorch (semana 8). Los constructores ya están escritos: los pesos empiezan al azar con un generador `rng` (así todos obtenemos los mismos números) y los sesgos en 0. `Module` trae `zero_grad`, que vuelve a 0 el gradiente de todos los parámetros.

**5a.** Completa:

- `Neuron.__call__(x)`: recibe una lista de números (o de `Value`) y devuelve $z = \sum_j w_j x_j + b$, con `.relu()` si `self.nonlin` es verdadero. Escribe `w * x`, con el `Value` a la izquierda.
- `Neuron.parameters()`: la lista de sus pesos más el sesgo.
- `Layer.__call__(x)`: la lista con la salida de cada neurona. `Layer.parameters()`: todos los parámetros de sus neuronas, en una lista plana.
- `MLP.__call__(x)`: pasa `x` por cada capa en orden. Si la última capa tiene una sola neurona, devuelve ese `Value` solo, no una lista. `MLP.parameters()`: todos, en una lista plana.

In [ ]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        return []


class Neuron(Module):
    def __init__(self, n_in, nonlin, rng):
        self.w = [Value(v) for v in rng.normal(0, 1 / math.sqrt(n_in), n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        # TODO 5a
        raise NotImplementedError

    def parameters(self):
        # TODO 5a
        raise NotImplementedError


class Layer(Module):
    def __init__(self, n_in, n_out, nonlin, rng):
        self.neurons = [Neuron(n_in, nonlin, rng) for _ in range(n_out)]

    def __call__(self, x):
        # TODO 5a
        raise NotImplementedError

    def parameters(self):
        # TODO 5a
        raise NotImplementedError


class MLP(Module):
    """MLP(2, [8, 8, 1], rng): 2 entradas, dos capas ocultas de 8 con ReLU y una salida sin activación."""

    def __init__(self, n_in, sizes, rng):
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1], nonlin=i < len(sizes) - 1, rng=rng) for i in range(len(sizes))]

    def __call__(self, x):
        # TODO 5a
        raise NotImplementedError

    def parameters(self):
        # TODO 5a
        raise NotImplementedError

**5b.** Antes de ejecutar la verificación, calcula a mano cuántos parámetros tiene `MLP(2, [8, 8, 1], rng)` y escríbelo aquí con la cuenta.

*Tu respuesta:*

-

In [ ]:
def torch_layers(model):
    """Los pesos de tu red como tensores de PyTorch: [(W, b)] por capa, que guardan gradiente."""
    return [(torch.tensor([[w.data for w in n.w] for n in layer.neurons], requires_grad=True),
             torch.tensor([n.b.data for n in layer.neurons], requires_grad=True)) for layer in model.layers]


def torch_forward(layers, X):
    A = X if isinstance(X, torch.Tensor) else torch.tensor(X)
    for i, (W, b) in enumerate(layers):
        A = A @ W.T + b
        if i < len(layers) - 1:
            A = torch.relu(A)
    return A[:, 0]


net = MLP(2, [8, 8, 1], np.random.default_rng(SEED))
params = net.parameters()
assert all(isinstance(p, Value) for p in params), "parameters() debe devolver una lista plana de Value"
assert len(params) == len({id(p) for p in params}), "parameters() repite algún parámetro"
n_params = len(params)
outs = [net(x) for x in Xl_train[:10]]
assert all(isinstance(o, Value) for o in outs), "con una sola neurona de salida, MLP debe devolver un Value, no una lista"
layers_t = torch_layers(net)
assert np.allclose([o.data for o in outs], torch_forward(layers_t, Xl_train[:10]).detach().numpy()), \
    "la ida no coincide con la misma red escrita con matrices"

# Una pérdida cualquiera, para comparar todos los gradientes con PyTorch
loss = sum(o ** 2 for o in outs) / len(outs)
net.zero_grad()
loss.backward()
loss_t = (torch_forward(layers_t, Xl_train[:10]) ** 2).mean()
loss_t.backward()
for k, (layer, (W, b)) in enumerate(zip(net.layers, layers_t)):
    assert np.allclose([[w.grad for w in n.w] for n in layer.neurons], W.grad.numpy()), f"los gradientes de W en la capa {k} no coinciden con PyTorch"
    assert np.allclose([n.b.grad for n in layer.neurons], b.grad.numpy()), f"los gradientes de b en la capa {k} no coinciden con PyTorch"
print(f"✓ la red tiene {n_params} parámetros, y su ida y sus gradientes coinciden con PyTorch")

**Guarda en GitHub** con el mensaje `Ejercicio 5 resuelto`.

## Ejercicio 6: entrenar sobre *two moons*

### La pérdida

La red da un puntaje $z$ y la probabilidad de la clase 1 es $p = \sigma(z)$. La pérdida es la entropía cruzada binaria. Escrita con $p$ tiene un problema práctico: si la red está muy segura, $\sigma(z)$ se redondea a exactamente 1.0 en la computadora, y $\ln(1 - p) = \ln 0$ da error. Con un poco de álgebra (usa $\ln \sigma(z) = z - \ln(1 + e^z)$ y $\ln(1 - \sigma(z)) = -\ln(1 + e^z)$) se llega a una forma equivalente que no tiene ese problema:

$$L = -\left[\, y \ln p + (1 - y) \ln(1 - p) \,\right] = \ln(1 + e^{z}) - y\,z$$

**6a.** Completa `bce_with_logits(z, y)` con la forma de la derecha, para un `Value` `z` y una etiqueta `y` que vale 0 o 1. (*Logits* es el nombre que se le da a los puntajes antes de la sigmoide.)

In [ ]:
def bce_with_logits(z, y):
    # TODO 6a
    raise NotImplementedError

In [ ]:
for zv in (-3.0, -0.2, 0.0, 1.7):
    for yv in (0, 1):
        z = Value(zv)
        out = bce_with_logits(z, yv)
        p = 1 / (1 + math.exp(-zv))
        assert math.isclose(out.data, -(yv * math.log(p) + (1 - yv) * math.log(1 - p)), rel_tol=1e-9), \
            f"con z = {zv}, y = {yv} no coincide con la entropía cruzada"
        out.backward()
        assert math.isclose(z.grad, p - yv, rel_tol=1e-9, abs_tol=1e-12), "la derivada debería ser p − y"
assert bce_with_logits(Value(40.0), 1).data < 1e-15, "con z = 40 y y = 1 la pérdida debe ser casi 0, sin error"
print("✓ bce_with_logits coincide con la entropía cruzada, y su derivada es p − y")

### El entrenamiento

**6b.** Completa `fit(model, X, y, steps, lr)`. En cada paso:

1. **Ida**: el puntaje de cada ejemplo, `model(x)`, y la pérdida **promedio** de `bce_with_logits` sobre todos los ejemplos.
2. **Vuelta**: `model.zero_grad()` y después `loss.backward()`.
3. **Paso**: para cada parámetro, `p.data -= lr * p.grad`.

Guarda en la lista `losses` la pérdida de cada paso (el número, `loss.data`) y en `accs` la *accuracy* de entrenamiento de ese paso (predice 1 si $z > 0$). Devuelve `losses, accs`.

Completa también `predict(model, X)`, que devuelve la lista de predicciones (0 o 1).

Cada paso recorre todo el entrenamiento con tu motor, número por número: espera alrededor de un cuarto de segundo por paso.

In [ ]:
def fit(model, X, y, steps, lr):
    # TODO 6b
    raise NotImplementedError


def predict(model, X):
    # TODO 6b
    raise NotImplementedError

In [ ]:
# Tres pasos sobre 20 ejemplos, comparados con los mismos tres pasos hechos con PyTorch
small = MLP(2, [4, 1], np.random.default_rng(1))
layers_t = torch_layers(small)
X_few, y_few = Xl_train[:20], yl_train[:20]
losses_few, accs_few = fit(small, X_few, y_few, steps=3, lr=0.5)
assert len(losses_few) == 3 and len(accs_few) == 3, "fit debe devolver una pérdida y una accuracy por paso"
for _ in range(3):
    loss_t = torch.nn.functional.binary_cross_entropy_with_logits(torch_forward(layers_t, X_few), torch.tensor(y_few, dtype=torch.float64))
    for W, b in layers_t:
        W.grad, b.grad = None, None
    loss_t.backward()
    with torch.no_grad():
        for W, b in layers_t:
            W -= 0.5 * W.grad
            b -= 0.5 * b.grad
for layer, (W, b) in zip(small.layers, layers_t):
    assert np.allclose([[w.data for w in n.w] for n in layer.neurons], W.detach().numpy()), \
        "después de 3 pasos tus pesos no coinciden con PyTorch: revisa el promedio, zero_grad y el signo del paso"
assert math.isclose(losses_few[0], torch.nn.functional.binary_cross_entropy_with_logits(
    torch_forward(torch_layers(MLP(2, [4, 1], np.random.default_rng(1))), X_few), torch.tensor(y_few, dtype=torch.float64)).item()), \
    "la primera pérdida guardada debe ser la de antes del primer paso"
preds_few = predict(small, X_few)
assert len(preds_few) == 20 and all(v in (0, 1) for v in preds_few), "predict devuelve una lista de 0 y 1, uno por ejemplo"
print("✓ fit da los mismos pesos que PyTorch después de 3 pasos")

**6c.** Ahora el entrenamiento de verdad: `MLP(2, [8, 8, 1])`, 120 pasos con tasa 1.5. Ejecuta la celda (tarda medio minuto o un minuto) y mira las dos curvas.

In [ ]:
model = MLP(2, [8, 8, 1], np.random.default_rng(SEED))
start = time.perf_counter()
losses, accs = fit(model, Xl_train, yl_train, steps=120, lr=1.5)
t_fit = time.perf_counter() - start
print(f"120 pasos en {t_fit:.0f} s ({t_fit / 120:.2f} s por paso)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(losses, color="tab:blue")
axes[0].axhline(math.log(2), color="gray", linestyle="--", label="adivinar: ln 2")
axes[0].set(xlabel="paso", ylabel="entropía cruzada", title="La pérdida")
axes[0].legend(fontsize=8)
axes[1].plot(accs, color="tab:green")
axes[1].set(xlabel="paso", ylabel="accuracy de entrenamiento", title="La accuracy", ylim=(0.4, 1.02))
plt.tight_layout()
plt.show()

In [ ]:
acc_mine = np.mean(np.array(predict(model, Xl_test)) == y_test)
assert losses[-1] < 0.25, f"la pérdida final debería bajar de 0.25; es {losses[-1]:.3f}"
assert acc_mine >= 0.9, f"la accuracy de prueba debería pasar de 0.9; es {acc_mine:.3f}"
print(f"✓ tu red: pérdida final {losses[-1]:.3f}, accuracy de prueba {acc_mine:.3f}")

**6d.** La comparación de siempre, sobre la misma partición: la línea base, la regresión logística (una recta) y `MLPClassifier` de sklearn con la misma forma, dos capas ocultas de 8. Con tan pocos datos, el resultado de una red depende bastante de sus pesos iniciales, así que `MLPClassifier` se entrena con 5 semillas y se muestra el promedio y el rango. Después, las regiones que dibuja cada modelo. Ejecuta las dos celdas.

In [ ]:
others = {
    "línea base (la clase más común)": DummyClassifier(strategy="most_frequent").fit(X_train, y_train),
    "regresión logística": LogisticRegression().fit(X_train, y_train),
}
sk_mlps = [MLPClassifier((8, 8), max_iter=3000, random_state=s).fit(X_train, y_train) for s in range(5)]
sk_scores = [m.score(X_test, y_test) for m in sk_mlps]
print(f"{'modelo':<34}{'prueba':>8}")
for name, m in others.items():
    print(f"{name:<34}{m.score(X_test, y_test):>8.3f}")
print(f"{'MLPClassifier (8, 8), 5 semillas':<34}{np.mean(sk_scores):>8.3f}   (de {min(sk_scores):.2f} a {max(sk_scores):.2f})")
print(f"{'tu MLP (2, [8, 8, 1])':<34}{acc_mine:>8.3f}")

In [ ]:
gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 40), np.linspace(-1.1, 1.6, 40))
grid = np.c_[gx.ravel(), gy.ravel()]
regions = {
    "regresión logística": others["regresión logística"].predict(grid),
    "MLPClassifier (8, 8), semilla 0": sk_mlps[0].predict(grid),
    "tu MLP": np.array(predict(model, grid.tolist())),
}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (name, region) in zip(axes, regions.items()):
    ax.contourf(gx, gy, region.reshape(gx.shape), levels=[-0.5, 0.5, 1.5], colors=["tab:blue", "tab:orange"], alpha=0.2)
    for c, color in [(0, "tab:blue"), (1, "tab:orange")]:
        ax.scatter(*X_test[y_test == c].T, s=16, color=color, edgecolor="white", label=f"clase {c}")
    ax.set(title=name, xlabel="$x_1$")
axes[0].set_ylabel("$x_2$")
axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Guarda en GitHub** con el mensaje `Ejercicio 6 resuelto`.

## Ejercicio 7: tres experimentos

### 7a. Olvidar `zero_grad`

`backward` **suma** a lo que ya hay en `.grad` (el `+=` del ejercicio 1). Si no lo reinicias, cada paso usa la suma de todos los gradientes anteriores.

Escribe `fit_forgetful`: una copia de tu `fit` **sin** `model.zero_grad()`. Entrena con ella una red nueva igual a la de 6c durante 40 pasos con tasa 0.5, y otra red igual con tu `fit` normal y los mismos 40 pasos con tasa 0.5. Dibuja las dos curvas de pérdida en un mismo gráfico, con leyenda. Guarda las pérdidas en `losses_ok` y `losses_forgetful`.

In [ ]:
# TODO 7a
losses_ok, losses_forgetful = None, None

In [ ]:
assert losses_ok is not None and losses_forgetful is not None, "guarda las dos listas de pérdidas"
assert len(losses_ok) == 40 and len(losses_forgetful) == 40, "40 pasos cada una"
assert losses_ok[0] == losses_forgetful[0], "las dos redes deben empezar iguales (misma semilla)"
assert losses_forgetful[1] < losses_forgetful[0], "el primer paso es igual en las dos: la pérdida baja"
assert losses_ok[-1] < 0.35, "con zero_grad y tasa 0.5, la pérdida debería bajar de 0.35 en 40 pasos"
assert max(losses_forgetful) > 1.5 * losses_ok[0], "sin zero_grad la pérdida debería llegar a subir muy por encima de donde empezó"
print("✓ las dos curvas")

**7a (pregunta).** Describe las dos curvas con números. ¿Por qué el primer paso es idéntico en las dos? ¿Qué le pasa al tamaño de los pasos a medida que se acumulan los gradientes, y cómo explica eso la forma de la curva naranja?

*Tu respuesta:*

-

### 7b. ¿Cuántos nodos tiene una pérdida?

**7b.** Escribe `count_nodes(root)`: el número de `Value` **distintos** del grafo de `root` (los mismos que recorre tu `backward`). Úsala para contar los nodos de la pérdida de **un** paso de entrenamiento de la red de 6c sobre los 150 ejemplos, y guarda el número en `n_nodes`.

In [ ]:
def count_nodes(root):
    # TODO 7b
    raise NotImplementedError


n_nodes = None  # TODO 7b

In [ ]:
a = Value(2.0)
assert count_nodes(a) == 1, "un número solo es un nodo"
assert count_nodes(a * a) == 2, "a · a tiene dos nodos: a y el resultado (a cuenta una vez)"
assert count_nodes((a + 1) * (a + 1)) == 6, "(a + 1)(a + 1) tiene 6: a, los dos 1, las dos sumas y el producto"
assert n_nodes is not None and 25_000 < n_nodes < 45_000, "n_nodes debería contar la pérdida de un paso sobre los 150 ejemplos"
print(f"✓ count_nodes: {n_nodes} nodos en la pérdida de un paso")

### 7c. Tu motor contra PyTorch

Ejecuta la celda: mide el tiempo de una ida y una vuelta sobre los 150 ejemplos con tu motor, y lo mismo con PyTorch, con los mismos pesos.

In [ ]:
bench = MLP(2, [8, 8, 1], np.random.default_rng(SEED))
start = time.perf_counter()
loss_b = sum(bce_with_logits(bench(x), yi) for x, yi in zip(Xl_train, yl_train)) / len(yl_train)
bench.zero_grad()
loss_b.backward()
t_mine = time.perf_counter() - start

torch.set_num_threads(1)  # con matrices tan chicas, repartir el trabajo entre varios núcleos cuesta más de lo que ahorra
layers_b = torch_layers(bench)
X_t, y_t = torch.tensor(Xl_train), torch.tensor(yl_train, dtype=torch.float64)
start = time.perf_counter()
for _ in range(100):
    for W, b in layers_b:
        W.grad, b.grad = None, None
    torch.nn.functional.binary_cross_entropy_with_logits(torch_forward(layers_b, X_t), y_t).backward()
t_torch = (time.perf_counter() - start) / 100

print(f"tu motor: {1000 * t_mine:.1f} ms   PyTorch: {1000 * t_torch:.2f} ms   PyTorch es unas {t_mine / t_torch:.0f} veces más rápido")
print("¿mismos gradientes?", np.allclose(bench.layers[0].neurons[0].w[0].grad, layers_b[0][0].grad[0, 0].item()))

**7c (pregunta).** Los dos calculan los mismos gradientes con el mismo algoritmo. ¿De dónde sale la diferencia de tiempo? Usa tu `n_nodes` en la respuesta: ¿cuántos objetos de Python crea tu motor en cada paso, y cuántas operaciones hace PyTorch para la misma cuenta, si trabaja con matrices como la semana 7?

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 7 resuelto`.

## Ejercicio 8: interpretar

Responde con tus palabras y tus números, en dos a cuatro oraciones cada una.

1. **El `+=`.** Da un ejemplo concreto, con números, donde escribir `self.grad = ...` en vez de `self.grad += ...` en un `_backward` da un gradiente equivocado. ¿Qué verificación de este trabajo lo habría detectado?
2. **El orden.** Imagina que `backward` recorre los nodos en un orden cualquiera. En la función `diamond` del ejercicio 2, ¿qué nodo podría repartir su gradiente antes de tiempo, y qué le pasaría al resultado?
3. **A mano contra automático.** Compara la semana 7, donde escribimos la vuelta a mano con matrices, con tu motor. ¿Qué es igual? ¿Qué ganas con el motor, y qué pierdes?
4. **Tu red.** Con tu tabla y tus regiones de 6d: ¿cuánto mejora tu red sobre la regresión logística, y por qué una recta no alcanza en este conjunto? ¿Qué probarías para mejorarla?

*Tus respuestas:*

- 1.
- 2.
- 3.
- 4.

**Guarda en GitHub** con el mensaje `Ejercicio 8 resuelto`.

## Conclusión

*Escribe aquí un párrafo en lenguaje sencillo: qué hace tu motor, qué significan tus números comparados con la línea base, la regresión logística y sklearn, y qué te gustaría probar después. Sin este párrafo el trabajo no tiene nota.*

## Uso de IA

*Una línea: para qué usaste IA en este trabajo, o "No usé IA".*

## Rúbrica

Primero asígnate tu puntaje (columna "Yo"). Tu compañero revisor completa la columna "Revisor" con un comentario en la copia de Drive que le compartas.

| Criterio | Qué se mira | Yo (0–2) | Revisor (0–2) |
|---|---|---|---|
| **Se ejecuta** | *Entorno de ejecución → Reiniciar y ejecutar todo* corre sin errores | | |
| **Es correcto** | Todas las verificaciones imprimen ✓; ninguna derivada está escrita a mano fuera de los `_backward` de las operaciones (el ejercicio 4 usa tu motor); el gráfico de 7a tiene las dos curvas y leyenda | | |
| **Está justificado** | Las respuestas de 4b, 5b, 7a, 7c y del ejercicio 8 explican el porqué con tus números | | |
| **Está bien comunicado** | La conclusión se entiende sin leer el código; hay un *commit* por ejercicio con un mensaje claro | | |
| **Total** | | /8 | /8 |

**Antes de la clase de la semana 8**: ejecuta *Entorno de ejecución → Reiniciar y ejecutar todo*, revisa que todo imprima ✓ y guarda por última vez en `P1/P1_autograd_desde_cero.ipynb`.